In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv('users_commodities_india.csv')
df['commodity_list'] = df['commodity'].apply(
    lambda x: [c.strip() for c in x.split(';')]
)
print(df.shape)
df.head()

(1000, 8)


,user_id,commodity,role,city,state,min_quantity_mt,max_quantity_mt,commodity_list
0,1,sugar;rice,trader,Mumbai,Maharashtra,43,216,"[sugar, rice]"
1,2,sugar;rice,trader,Indore,Madhya Pradesh,74,191,"[sugar, rice]"
2,3,sugar;rice,exporter,Satna,Madhya Pradesh,79,298,"[sugar, rice]"
3,4,rice;sugar;cotton,trader,Salem,Tamil Nadu,59,257,"[rice, sugar, cotton]"
4,5,rice;sugar,exporter,Suryapet,Telangana,76,320,"[rice, sugar]"


In [2]:
ALL_COMMODITIES = sorted(set(
    c for lst in df['commodity_list'] for c in lst
))
print("Commodities found:", ALL_COMMODITIES)

COMMODITY_BOOST = 0.9

def encode_commodity(commodity_list):
    vec = np.zeros(len(ALL_COMMODITIES))
    for c in commodity_list:
        if c in ALL_COMMODITIES:
            vec[ALL_COMMODITIES.index(c)] = 1.0
    return vec * COMMODITY_BOOST

commodity_vecs = np.array([
    encode_commodity(row['commodity_list'])
    for _, row in df.iterrows()
])

print("Dimension names:", [f"has_{c}" for c in ALL_COMMODITIES])
print()
for i in range(5):
    row = df.iloc[i]
    raw = commodity_vecs[i] / COMMODITY_BOOST
    vals = dict(zip([f"has_{c}" for c in ALL_COMMODITIES], raw))
    print(f"User {row['user_id']} | {row['commodity']:20s} → {vals}")

Commodities found: ['cotton', 'rice', 'sugar']
Dimension names: ['has_cotton', 'has_rice', 'has_sugar']

User 1 | sugar;rice           → {'has_cotton': np.float64(0.0), 'has_rice': np.float64(1.0), 'has_sugar': np.float64(1.0)}
User 2 | sugar;rice           → {'has_cotton': np.float64(0.0), 'has_rice': np.float64(1.0), 'has_sugar': np.float64(1.0)}
User 3 | sugar;rice           → {'has_cotton': np.float64(0.0), 'has_rice': np.float64(1.0), 'has_sugar': np.float64(1.0)}
User 4 | rice;sugar;cotton    → {'has_cotton': np.float64(1.0), 'has_rice': np.float64(1.0), 'has_sugar': np.float64(1.0)}
User 5 | rice;sugar           → {'has_cotton': np.float64(0.0), 'has_rice': np.float64(1.0), 'has_sugar': np.float64(1.0)}


In [25]:
ROLE_AFFINITY = {
    'trader':   {'want_broker': 0.55, 'want_exporter': 0.3, 'want_trader': 0.2},
    'broker':   {'want_broker': 0.33,'want_exporter': 0.33,'want_trader': 0.33},
    'exporter': {'want_broker': 0.7, 'want_exporter': 0.2, 'want_trader': 0.3},
}

# What each role OFFERS — used for candidate vectors
ROLE_OFFERS = {
    'trader':   {'is_broker': 0.0, 'is_exporter': 0.0, 'is_trader': 1.0},
    'broker':   {'is_broker': 1.0, 'is_exporter': 0.0, 'is_trader': 0.0},
    'exporter': {'is_broker': 0.0, 'is_exporter': 1.0, 'is_trader': 0.0},
}

ROLE_DIMS  = ['want_broker', 'want_exporter', 'want_trader']
ROLE_BOOST = 1.5

def encode_role_searcher(role):
    """What this user WANTS — used for the query vector."""
    affinity = ROLE_AFFINITY[role]
    return np.array([affinity[d] for d in ROLE_DIMS]) * ROLE_BOOST

def encode_role_candidate(role):
    offers = ROLE_OFFERS[role]
    return np.array([
        offers['is_broker'],
        offers['is_exporter'],
        offers['is_trader']
    ]) * ROLE_BOOST

# Stored vectors use candidate encoding (what they ARE)
role_vecs = np.array([
    encode_role_candidate(row['role'])
    for _, row in df.iterrows()
])

print("Dimension names:", ROLE_DIMS)
print()
print("Candidate vectors (what they ARE — stored in DB):")
for role in ['trader', 'broker', 'exporter']:
    v = encode_role_candidate(role) / ROLE_BOOST
    print(f"  {role:10s} → {dict(zip(ROLE_DIMS, v))}")

print()
print("Searcher vectors (what they WANT — used at query time only):")
for role in ['trader', 'broker', 'exporter']:
    v = encode_role_searcher(role) / ROLE_BOOST
    print(f"  {role:10s} → {dict(zip(ROLE_DIMS, v))}")

print()
print("Dot product preview for trader searching:")
trader_want = encode_role_searcher('trader')
for role in ['broker', 'exporter', 'trader']:
    candidate_offer = encode_role_candidate(role)
    dot = np.dot(trader_want, candidate_offer)
    print(f"  trader → {role}: dot = {dot:.3f}")

Dimension names: ['want_broker', 'want_exporter', 'want_trader']

Candidate vectors (what they ARE — stored in DB):
  trader     → {'want_broker': np.float64(0.0), 'want_exporter': np.float64(0.0), 'want_trader': np.float64(1.0)}
  broker     → {'want_broker': np.float64(1.0), 'want_exporter': np.float64(0.0), 'want_trader': np.float64(0.0)}
  exporter   → {'want_broker': np.float64(0.0), 'want_exporter': np.float64(1.0), 'want_trader': np.float64(0.0)}

Searcher vectors (what they WANT — used at query time only):
  trader     → {'want_broker': np.float64(0.55), 'want_exporter': np.float64(0.3), 'want_trader': np.float64(0.20000000000000004)}
  broker     → {'want_broker': np.float64(0.33), 'want_exporter': np.float64(0.33), 'want_trader': np.float64(0.33)}
  exporter   → {'want_broker': np.float64(0.6999999999999998), 'want_exporter': np.float64(0.20000000000000004), 'want_trader': np.float64(0.3)}

Dot product preview for trader searching:
  trader → broker: dot = 1.238
  trader → ex

In [26]:
city_enc  = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
state_enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

city_vecs  = city_enc.fit_transform(df[['city']])
state_vecs = state_enc.fit_transform(df[['state']])

print("City dimensions:",  city_enc.categories_[0].tolist())
print("State dimensions:", state_enc.categories_[0].tolist())
print()
print("City vec shape:", city_vecs.shape)
print("State vec shape:", state_vecs.shape)

City dimensions: ['Adilabad', 'Agra', 'Ahmedabad', 'Ajmer', 'Alwar', 'Amravati', 'Amritsar', 'Anand', 'Asansol', 'Aurangabad', 'Ballari', 'Bareilly', 'Bathinda', 'Belagavi', 'Bengaluru', 'Bharatpur', 'Bhavnagar', 'Bhopal', 'Bikaner', 'Chennai', 'Coimbatore', 'Davangere', 'Dindigul', 'Durgapur', 'Erode', 'Gandhinagar', 'Ghaziabad', 'Gorakhpur', 'Gwalior', 'Haldia', 'Hoshiarpur', 'Howrah', 'Hubli', 'Hyderabad', 'Indore', 'Jabalpur', 'Jaipur', 'Jalandhar', 'Jalgaon', 'Jamnagar', 'Jodhpur', 'Junagadh', 'Kanpur', 'Karimnagar', 'Katni', 'Khammam', 'Kharagpur', 'Kolhapur', 'Kolkata', 'Kota', 'Krishnanagar', 'Lucknow', 'Ludhiana', 'Madurai', 'Mahbubnagar', 'Malda', 'Mangaluru', 'Meerut', 'Miryalaguda', 'Moga', 'Mohali', 'Mumbai', 'Mysuru', 'Nagpur', 'Nanded', 'Nashik', 'Navsari', 'Nizamabad', 'Noida', 'Pali', 'Pathankot', 'Patiala', 'Phagwara', 'Prayagraj', 'Pune', 'Raiganj', 'Rajkot', 'Ramagundam', 'Ratlam', 'Rewa', 'Sagar', 'Salem', 'Satna', 'Shivamogga', 'Sikar', 'Siliguri', 'Solapur', 'Sur

In [27]:
# Final vector layout:
# [ has_cotton | has_rice | has_sugar | want_broker | want_exporter | want_trader | city_onehot... | state_onehot... ]

master_vectors = np.hstack([
    commodity_vecs,   # explicitly named has_X dims
    role_vecs,        # explicitly named want_X dims
    # city_vecs,
    state_vecs,
])

df['vector'] = master_vectors.tolist()

# Print dimension layout so you know exactly what each position means
n_comm  = commodity_vecs.shape[1]
n_role  = role_vecs.shape[1]
# n_city  = city_vecs.shape[1]
n_state = state_vecs.shape[1]
total   = master_vectors.shape[1]

print(f"Total vector dims: {total}")
print(f"  [0 : {n_comm}]              → commodity  ({[f'has_{c}' for c in ALL_COMMODITIES]})")
print(f"  [{n_comm} : {n_comm+n_role}]          → role       ({ROLE_DIMS})")
print(f"  [{n_comm+n_role} : {n_comm+n_role}]   → city       ({city_enc.categories_[0].tolist()})")
print(f"  [{n_comm+n_role} : {total}]  → state      ({state_enc.categories_[0].tolist()})")

Total vector dims: 16
  [0 : 3]              → commodity  (['has_cotton', 'has_rice', 'has_sugar'])
  [3 : 6]          → role       (['want_broker', 'want_exporter', 'want_trader'])
  [6 : 6]   → city       (['Adilabad', 'Agra', 'Ahmedabad', 'Ajmer', 'Alwar', 'Amravati', 'Amritsar', 'Anand', 'Asansol', 'Aurangabad', 'Ballari', 'Bareilly', 'Bathinda', 'Belagavi', 'Bengaluru', 'Bharatpur', 'Bhavnagar', 'Bhopal', 'Bikaner', 'Chennai', 'Coimbatore', 'Davangere', 'Dindigul', 'Durgapur', 'Erode', 'Gandhinagar', 'Ghaziabad', 'Gorakhpur', 'Gwalior', 'Haldia', 'Hoshiarpur', 'Howrah', 'Hubli', 'Hyderabad', 'Indore', 'Jabalpur', 'Jaipur', 'Jalandhar', 'Jalgaon', 'Jamnagar', 'Jodhpur', 'Junagadh', 'Kanpur', 'Karimnagar', 'Katni', 'Khammam', 'Kharagpur', 'Kolhapur', 'Kolkata', 'Kota', 'Krishnanagar', 'Lucknow', 'Ludhiana', 'Madurai', 'Mahbubnagar', 'Malda', 'Mangaluru', 'Meerut', 'Miryalaguda', 'Moga', 'Mohali', 'Mumbai', 'Mysuru', 'Nagpur', 'Nanded', 'Nashik', 'Navsari', 'Nizamabad', 'Noida', 'Pal

In [28]:
# Quantity stored as raw metadata only — used for DB filter, not in vector
vector_db = []

for idx, row in df.iterrows():
    vector_db.append({
        "id":     int(row['user_id']),
        "vector": master_vectors[idx],
        "meta": {
            "user_id":        int(row['user_id']),
            "role":           row['role'],
            "commodity":      row['commodity'],
            "commodity_list": row['commodity_list'],
            "city":           row['city'],
            "state":          row['state'],
            "qty_min":        int(row['min_quantity_mt']),
            "qty_max":        int(row['max_quantity_mt']),
        }
    })

print(f"Vector DB: {len(vector_db)} records")
print("\nSample:")
r = vector_db[0]
print(f"  id: {r['id']}")
print(f"  meta: {r['meta']}")
print(f"  vector dims: {len(r['vector'])}")
print(f"  vector (commodity+role section): {r['vector'][:n_comm+n_role]}")
print(f"  names: {[f'has_{c}' for c in ALL_COMMODITIES] + ROLE_DIMS}")

Vector DB: 1000 records

Sample:
  id: 1
  meta: {'user_id': 1, 'role': 'trader', 'commodity': 'sugar;rice', 'commodity_list': ['sugar', 'rice'], 'city': 'Mumbai', 'state': 'Maharashtra', 'qty_min': 43, 'qty_max': 216}
  vector dims: 16
  vector (commodity+role section): [0.  0.9 0.9 0.  0.  1.5]
  names: ['has_cotton', 'has_rice', 'has_sugar', 'want_broker', 'want_exporter', 'want_trader']


In [29]:
def search(searcher_id, top_k=50):
    searcher = next(r for r in vector_db if r['meta']['user_id'] == searcher_id)
    searcher_role = searcher['meta']['role']

    # Build query vector with WANT encoding — not what's stored in DB
    s_commodity = encode_commodity(searcher['meta']['commodity_list'])
    s_role      = encode_role_searcher(searcher_role)   # WANT dims
    s_state     = state_enc.transform([[searcher['meta']['state']]])[0]

    s_vec = np.concatenate([s_commodity, s_role, s_state]).reshape(1, -1)

    # All users except searcher
    candidates = [
        r for r in vector_db
        if r['meta']['user_id'] != searcher_id
    ]

    if not candidates:
        print("No candidates found.")
        return pd.DataFrame()

    # Stored vectors use IS (offer) encoding — query uses WANT encoding
    # Dot product is now high when want_broker × is_broker = 0.5×4 × 1.0×4 = 8.0
    # vs want_trader × is_trader = 0.2×4 × 1.0×4 = 3.2
    cand_vecs = np.array([r['vector'] for r in candidates])
    sims = cosine_similarity(s_vec, cand_vecs)[0]

    top_idx = np.argsort(sims)[::-1][:top_k]

    results = []
    for i in top_idx:
        m = candidates[i]['meta']
        results.append({
            'user_id':    m['user_id'],
            'role':       m['role'],
            'commodity':  m['commodity'],
            'city':       m['city'],
            'state':      m['state'],
            'qty_range':  f"{m['qty_min']}–{m['qty_max']}mt",
            'similarity': round(float(sims[i]), 4),
        })

    return pd.DataFrame(results)


In [31]:
TEST_USER_ID = 53

s = next(r['meta'] for r in vector_db if r['meta']['user_id'] == TEST_USER_ID)
print(f"Searcher: User {s['user_id']} | {s['role']} | {s['commodity']} | {s['state']} | {s['qty_min']}–{s['qty_max']}mt")
print("=" * 70)

results = search(TEST_USER_ID, top_k=15)
print(results.to_string(index=False))

print("\nWhat to verify:")
print("1. Same commodity users rank highest")
print("2. For a trader: brokers outrank exporters, exporters outrank traders")
print("3. Same city users rank above different city (same commodity+role)")
print("4. Anyone with zero qty overlap is absent from results entirely")

Searcher: User 53 | trader | sugar | Tamil Nadu | 32–225mt
 user_id     role         commodity            city      state qty_range  similarity
     212   broker      cotton;sugar           Erode Tamil Nadu   32–96mt      0.8278
     835   broker      cotton;sugar         Madurai Tamil Nadu  80–336mt      0.8278
      71   broker      cotton;sugar         Chennai Tamil Nadu  41–204mt      0.8278
     610   broker        rice;sugar     Thoothukudi Tamil Nadu  32–286mt      0.8278
      92   broker      sugar;cotton        Dindigul Tamil Nadu  59–278mt      0.8278
     595   broker      sugar;cotton        Dindigul Tamil Nadu  95–366mt      0.8278
     905   broker        sugar;rice Tiruchirappalli Tamil Nadu  21–190mt      0.8278
     375   broker rice;cotton;sugar         Chennai Tamil Nadu  30–120mt      0.7665
     362   broker sugar;cotton;rice         Madurai Tamil Nadu  21–171mt      0.7665
     465   broker cotton;rice;sugar      Coimbatore Tamil Nadu  68–360mt      0.7665
     3

c:\Users\Admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
